In [0]:
from pyspark.sql import functions as F

# ==========================================
# CONFIGURACIÓN
# ==========================================

TABLA_BRONZE = "quality_data.bronze.control_calidad"

TABLA_SILVER = "quality.silver.control_calidad_limpio"

# Cargar datos Bronze
df_bronze = spark.table(TABLA_BRONZE)

print("=" * 60)
print("CARGA DE BRONZE")
print("=" * 60)
print(f"Tabla origen: {TABLA_BRONZE}")
print(f"Registros: {df_bronze.count():,}")
print(f"Columnas: {len(df_bronze.columns)}")

display(df_bronze.limit(1286058))

In [0]:
# ==========================================
# RENOMBRAR COLUMNAS
# ==========================================

mapeo_columnas = {
    "HDR": "hdr",
    "Cliente": "cliente_codigo",
    "Control_Calidad": "control_calidad",
    "Variable": "variable_codigo",
    "Resultado": "resultado_original",
    "VariableNom": "variable_nombre",
    "ARTICULO": "articulo_codigo",
    "BarColNom": "color_nombre",
    "BarColNum": "color_codigo",
    "CCOpeCod": "operacion_codigo",
    "CCFchUti": "fecha_medicion",
    "CCvalFloat": "resultado_float_origen"
}

df_silver = df_bronze

columnas_no_encontradas = []

for columna_origen, columna_destino in mapeo_columnas.items():
    if columna_origen in df_silver.columns:
        if columna_destino not in df_silver.columns:
            df_silver = df_silver.withColumnRenamed(
                columna_origen,
                columna_destino
            )
    else:
        columnas_no_encontradas.append(columna_origen)

print("Columnas no encontradas en Bronze:")
print(columnas_no_encontradas)

print("\nColumnas después del renombramiento:")

for columna in df_silver.columns:
    print(columna)

In [0]:
# ==========================================
# CREAR COLUMNAS FALTANTES
# ==========================================

columnas_esperadas = {
    "hdr": "string",
    "cliente_codigo": "string",
    "control_calidad": "string",
    "variable_codigo": "string",
    "variable_nombre": "string",
    "resultado_original": "string",
    "resultado_float_origen": "double",
    "fecha_medicion": "timestamp",
    "articulo_codigo": "string",
    "color_codigo": "string",
    "color_nombre": "string",
    "operacion_codigo": "string",
   
}

columnas_creadas = []

for columna, tipo_dato in columnas_esperadas.items():
    if columna not in df_silver.columns:
        df_silver = df_silver.withColumn(
            columna,
            F.lit(None).cast(tipo_dato)
        )

        columnas_creadas.append(columna)

print("Columnas creadas porque no existían:")
print(columnas_creadas)

In [0]:
columnas_texto_deseadas = [
    "hdr",
    "cliente_codigo",
    "control_calidad",
    "variable_codigo",
    "variable_nombre",
    "articulo_codigo",
    "operacion_codigo",

    
]

# Procesar solamente columnas que realmente existen
columnas_texto_existentes = [
    columna
    for columna in columnas_texto_deseadas
    if columna in df_silver.columns
]


def limpiar_texto(nombre_columna):
    texto_limpio = F.regexp_replace(
        F.trim(F.col(nombre_columna).cast("string")),
        r"\s+",
        " "
    )

    return (
        F.when(
            F.col(nombre_columna).isNull()
            | (texto_limpio == ""),
            F.lit(None).cast("string")
        )
        .otherwise(
            F.upper(texto_limpio)
        )
    )


for columna in columnas_texto_existentes:
    df_silver = df_silver.withColumn(
        columna,
        limpiar_texto(columna)
    )

# ==========================================
# # Estandarizar el codigo del articulo protegiendo le privacidad de la empresa 
# ==========================================

if "articulo_codigo" in df_silver.columns:
    df_silver = df_silver.withColumn(
        "articulo_codigo",
        F.regexp_replace(
            F.col("articulo_codigo"),
            r"^PUU",
            "ART"
        )
    )

print("✅ Transformación aplicada correctamente")
print("Columnas procesadas:")
print(columnas_texto_existentes)

display(
    df_silver.select(
        "hdr",
        "cliente_codigo",
        "articulo_codigo"
    ).limit(30)
)

In [0]:
# ==========================================
# Estandarizar el codigo del articulo protegiendo le privacidad de la empresa 
# ==========================================

if "articulo_codigo" in df_silver.columns:
    df_silver = df_silver.withColumn(
        "articulo_codigo",
        F.regexp_replace(
            F.col("articulo_codigo"),
            r"^SSS",
            "ART"
        )
    )

print("✅ Transformación aplicada correctamente")
print("Columnas procesadas:")
print(columnas_texto_existentes)

display(
    df_silver.select(
        "hdr",
        "cliente_codigo",
        "articulo_codigo"
    ).limit(30)
)

In [0]:
# ==========================================
# NORMALIZACIÓN DE FECHAS
# ==========================================

df_silver = (
    df_silver
    .withColumn(
        "fecha_medicion",
        F.expr("try_cast(fecha_medicion AS TIMESTAMP)")
    )
    .withColumn(
        "fecha_medicion_dia",
        F.to_date("fecha_medicion")
    )
    .withColumn(
        "anio_medicion",
        F.year("fecha_medicion")
    )
    .withColumn(
        "mes_medicion",
        F.month("fecha_medicion")
    )
    .withColumn(
        "dia_medicion",
        F.dayofmonth("fecha_medicion")
    )
    .withColumn(
        "semana_medicion",
        F.weekofyear("fecha_medicion")
    )
)

display(
    df_silver.select(
        "fecha_medicion",
        "fecha_medicion_dia",
        "anio_medicion",
        "mes_medicion",
        "semana_medicion"
    ).limit(20)
)

In [0]:
# ==========================================
# ELIMINAR RESULTADOS VACÍOS
# ==========================================

total_antes_resultados = df_silver.count()

df_silver = df_silver.filter(
    F.col("resultado_original").isNotNull()
    & (F.trim(F.col("resultado_original").cast("string")) != "")
)

total_despues_resultados = df_silver.count()
resultados_vacios_eliminados = (
    total_antes_resultados - total_despues_resultados
)

print("=" * 60)
print("ELIMINACIÓN DE RESULTADOS VACÍOS")
print("=" * 60)
print(f"Registros antes: {total_antes_resultados:,}")
print(f"Registros después: {total_despues_resultados:,}")
print(f"Resultados vacíos eliminados: {resultados_vacios_eliminados:,}")

In [0]:
# ==========================================
# NORMALIZACIÓN DEL RESULTADO
# ==========================================

df_silver = (
    df_silver
    .withColumn(
        "resultado_texto",
        F.upper(
            F.trim(
                F.col("resultado_original").cast("string")
            )
        )
    )
    .withColumn(
        "_operador_extraido",
        F.regexp_extract(
            F.col("resultado_texto"),
            r"^(<=|>=|<|>|=)",
            1
        )
    )
    .withColumn(
        "operador_resultado",
        F.when(
            F.col("_operador_extraido") == "",
            F.lit(None).cast("string")
        ).otherwise(
            F.col("_operador_extraido")
        )
    )
    .withColumn(
        "_resultado_decimal",
        F.regexp_replace(
            F.col("resultado_texto"),
            ",",
            "."
        )
    )
    .withColumn(
        "_numero_extraido",
        F.regexp_extract(
            F.col("_resultado_decimal"),
            r"[-+]?[0-9]+(?:\.[0-9]+)?",
            0
        )
    )
    .withColumn(
        "resultado_numerico",
        F.coalesce(
            F.expr(
                "try_cast(resultado_float_origen AS DOUBLE)"
            ),
            F.when(
                F.col("_numero_extraido") != "",
                F.col("_numero_extraido").cast("double")
            )
        )
    )
    .drop(
        "_operador_extraido",
        "_resultado_decimal",
        "_numero_extraido"
    )
)

display(
    df_silver.select(
        "resultado_original",
        "resultado_texto",
        "resultado_float_origen",
        "resultado_numerico"
    ).limit(50)
)

In [0]:
# ==========================================
# CLASIFICACIÓN DEL RESULTADO
# ==========================================

df_silver = df_silver.withColumn(
    "tipo_resultado",
    F.when(
        F.col("resultado_numerico").isNotNull(),
        F.lit("NUMERICO")
    )
    .when(
        F.col("resultado_texto").isNull()
        | (F.col("resultado_texto") == ""),
        F.lit("VACIO")
    )
    .otherwise(
        F.lit("CATEGORICO")
    )
)

display(
    df_silver
    .groupBy("tipo_resultado")
    .agg(
        F.count("*").alias("cantidad")
    )
    .orderBy(F.desc("cantidad"))
)

In [0]:
# ==========================================
# CONTROL DE DUPLICADOS
# ==========================================

total_antes = df_silver.count()

df_silver_sin_duplicados = df_silver.dropDuplicates()

total_despues = df_silver_sin_duplicados.count()
duplicados_eliminados = total_antes - total_despues

print("=" * 60)
print("CONTROL DE DUPLICADOS")
print("=" * 60)
print(f"Registros antes: {total_antes:,}")
print(f"Registros después: {total_despues:,}")
print(f"Duplicados exactos eliminados: {duplicados_eliminados:,}")

In [0]:
# ==========================================
# CREAR TABLA SILVER SOLO CON REGISTROS APTOS PARA MODELO
# ==========================================

df_silver_modelable = (
    df_silver_sin_duplicados
    .filter(
        # HDR válido
        F.col("hdr").isNotNull()
        & (F.trim(F.col("hdr")) != "")

        # Variable válida
        & F.col("variable_codigo").isNotNull()
        & (F.trim(F.col("variable_codigo")) != "")

        # Fecha válida
        & F.col("fecha_medicion").isNotNull()

        # Resultado original no vacío
        & F.col("resultado_original").isNotNull()
        & (F.trim(F.col("resultado_original").cast("string")) != "")

        # Resultado numérico válido para modelo
        & F.col("resultado_numerico").isNotNull()
    )
)

print("=" * 60)
print("TABLA SILVER MODELABLE")
print("=" * 60)
print(f"Registros Bronze:                  {df_bronze.count():,}")
print(f"Registros después de duplicados:   {df_silver_sin_duplicados.count():,}")
print(f"Registros aptos para modelo:       {df_silver_modelable.count():,}")
print(f"Duplicados exactos eliminados:     {duplicados_eliminados:,}")

In [0]:
# ==========================================
# ELIMINAR COLUMNA para proteger datos
# ==========================================

columnas_a_eliminar = []

for columna in df_silver_modelable.columns:
    if columna.lower() in [
        "barserdsc",
        "articulo_nombre",
        "fascod",
        "fase_codigo",
        "procod",
        "proceso_codigo"
    ]:
        columnas_a_eliminar.append(columna)

print("Columnas que se eliminarán:")
print(columnas_a_eliminar)

df_silver_modelable = df_silver_modelable.drop(*columnas_a_eliminar)

print("Columnas finales de Silver:")
for columna in df_silver_modelable.columns:
    print(columna)

In [0]:
# ==========================================
# CREAR CATÁLOGO Y ESQUEMA SILVER
# ==========================================

spark.sql("CREATE CATALOG IF NOT EXISTS quality_data")
spark.sql("CREATE SCHEMA IF NOT EXISTS quality_data.silver")

print("✅ Esquema quality_data.silver disponible")


In [0]:

# ==========================================
# GUARDAR TABLA SILVER
# ==========================================

df_silver_modelable.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("quality_data.silver.control_calidad")

print("✅ Tabla Silver creada correctamente: quality_data.silver.control_calidad")

In [0]:
# ==========================================
# VALIDAR TABLA SILVER
# ==========================================

df_validacion = spark.table("quality_data.silver.control_calidad")

print(f"Registros: {df_validacion.count():,}")
print(f"Columnas: {len(df_validacion.columns)}")

display(df_validacion.limit(20))